# BRSR-QA - Stage 4: run S1, S2 and S3

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Put `stage4_bundle.zip` in the root of your Google Drive (My Drive).

Rough times on a T4: S1 or S2 over all 320 questions ~15-25 min each; QLoRA training ~30-50 min per fold; S3 inference ~15-25 min.
Everything is saved to Drive (`MyDrive/brsr_stage4`), so a disconnected session can be resumed: finished questions are skipped and finished adapters are kept.

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile
WORK = '/content/drive/MyDrive/brsr_stage4'
if not os.path.exists(WORK + '/src'):
    os.makedirs(WORK, exist_ok=True)
    zipfile.ZipFile('/content/drive/MyDrive/stage4_bundle.zip').extractall(WORK)
%cd {WORK}
!ls

In [ ]:
!pip -q install -U transformers peft bitsandbytes accelerate codecarbon pandas numpy

## 1. Check the plumbing (CPU, ~1 min)
Both lines should end with `passed`.

In [ ]:
!python tests/test_stage3.py | tail -2
!python tests/test_stage4.py | tail -2

## 2. S1 - retrieval + zero-shot
First a quick look at 8 questions, then the full run.

In [ ]:
!python src/run_inference.py --system S1 --folds 0 --limit 8 --out results/smoke_S1.jsonl

In [ ]:
import json
for line in list(open('results/smoke_S1.jsonl'))[:6]:
    r = json.loads(line); print(r['question_id'], '|', r['raw_output'][:120].replace('\n',' '), '| gold:')

In [ ]:
!python src/run_inference.py --system S1

## 3. S2 - retrieval + 3 in-prompt examples

In [ ]:
!python src/run_inference.py --system S2

## 4. S3 - fine-tune fold 0 first and look at it before training the rest
Step A is a 3-step smoke test (checks memory and that the loss is finite). Step B trains fold 0 properly.

In [ ]:
!python src/train_qlora.py --fold 0 --max-steps 3 --out adapters/_smoke --no-energy

In [ ]:
!python src/train_qlora.py --fold 0

In [ ]:
# answers fold-0 test companies with the fold-0 adapter, then prints accuracy / FAR / format failures
!python src/run_inference.py --system S3 --folds 0 --out results/predictions_S3_fold0_check.jsonl

In [ ]:
import json, collections
recs = [json.loads(l) for l in open('results/predictions_S3_fold0_check.jsonl')]
print(collections.Counter(r['parsed_answer']['kind'] for r in recs))
for r in recs[:10]: print(r['question_id'], '|', r['raw_output'][:100].replace('\n',' '))
# Red flags: (a) almost every answer is "not disclosed", (b) many parse errors / broken JSON.

## 5. If fold 0 looks sane: train folds 1-3 and run S3 on everything

In [ ]:
for fold in (1, 2, 3):
    !python src/train_qlora.py --fold {fold}

In [ ]:
!python src/run_inference.py --system S3

## 6. Compare the three systems

In [ ]:
import pandas as pd, sys
sys.path.insert(0, 'src')
from metrics import evaluate_jsonl, summarize
ann = pd.read_csv('data/annotations.csv', dtype=str).fillna('')
rows = []
for s in ('S1', 'S2', 'S3'):
    S = summarize(evaluate_jsonl(f'results/predictions_{s}_k5.jsonl', ann, system=s))
    rows.append({'system': s, 'n': S['n_scored'], 'accuracy': S['accuracy'], 'FAR': S['far'], 'UAR': S['uar'],
                 'format_fail': S['format_failure_rate'], 'parse_err': S['parse_error_rate']})
pd.DataFrame(rows).round(3)

Download or keep `results/` and `adapters/` (they are already in Drive). Do **not** upload the PDFs anywhere.